In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="kX0RjUlt2tVQzkpKm4ZV")
project = rf.workspace("hemavardhan").project("landmines-detection-dataset-sewvx")
version = project.version(1)
dataset = version.download("coco")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 133.9 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Landmines-detection-dataset-1 in coco:: 100%|██████████| 1202/1202 [00:00<00:00, 7890.62it/s]


In [ ]:
checkpoint_path = "/content/ssd_landmine_best.pth"


In [ ]:
import torch
from torchvision.models.detection import ssd300_vgg16, SSD300_VGG16_Weights
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model architecture (num_classes must match training)
weights = SSD300_VGG16_Weights.DEFAULT
model = ssd300_vgg16(weights=weights)
num_classes = 2
model.head.classification_head.num_classes = num_classes
model.to(device)

# Load trained weights
checkpoint_path = "/content/ssd_landmine_best.pth"
state_dict = torch.load(checkpoint_path, map_location=device)
missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
print("Missing keys:", missing_keys)
print("Unexpected keys:", unexpected_keys)

model.eval()

# Evaluation
VAL_ROOT = "/content/Landmines-detection-dataset-1/valid"
VAL_ANN  = "/content/Landmines-detection-dataset-1/valid/_annotations.coco.json"

from torchvision.datasets import CocoDetection
from torchvision import transforms

weights_transform = weights.transforms()

val_dataset = CocoDetection(root=VAL_ROOT, annFile=VAL_ANN, transform=lambda img: img)

preds = []
coco_gt = COCO(VAL_ANN)
with torch.no_grad():
    for idx in range(len(val_dataset)):
        img, _ = val_dataset[idx]
        img_t = weights_transform(img).to(device) if not isinstance(img, torch.Tensor) else img.to(device)
        output = model([img_t])[0]
        image_id = val_dataset.ids[idx] if hasattr(val_dataset, "ids") else coco_gt.getImgIds()[idx]
        boxes = output["boxes"].cpu().numpy()
        scores = output["scores"].cpu().numpy()
        labels = output["labels"].cpu().numpy()
        for box, score, label in zip(boxes, scores, labels):
            x1,y1,x2,y2 = box
            preds.append({
                "image_id": image_id,
                "category_id": int(label),
                "bbox": [float(x1), float(y1), float(x2-x1), float(y2-y1)],
                "score": float(score)
            })

# Save predictions
with open("ssd_preds.json", "w") as f:
    json.dump(preds, f)

# COCO evaluation
coco_dt = coco_gt.loadRes("ssd_preds.json")
coco_eval = COCOeval(coco_gt, coco_dt, iouType="bbox")
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()


Missing keys: []
Unexpected keys: []
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.06s).
Accumulating evaluation results...
DONE (t=0.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.423
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.817
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.379
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.333
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.518
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.650
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.456
 Average Recall     (AR) 

In [ ]:
import numpy as np

# Evaluate SSD on val_dataset using COCO
coco_gt = COCO(VAL_ANN)
coco_dt = coco_gt.loadRes("ssd_preds.json")
coco_eval = COCOeval(coco_gt, coco_dt, iouType="bbox")
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

# Extract metrics manually (approx)
precision = coco_eval.stats[0]  # mAP @0.5:0.95
recall    = coco_eval.stats[8]  # AR @ maxDets=100
map50     = coco_eval.stats[1]  # mAP @0.5
map5095   = coco_eval.stats[0]  # mAP @0.5:0.95

# YOLO-style fitness: weighted combination
fitness = 0.1*precision + 0.9*recall

print("\nSSD Metrics (approx):")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"mAP@0.5:  {map50:.4f}")
print(f"mAP@0.5:0.95: {map5095:.4f}")
print(f"Fitness (YOLO-style): {fitness:.4f}")


loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.08s).
Accumulating evaluation results...
DONE (t=0.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.423
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.817
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.379
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.333
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.518
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.650
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.456
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.491
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets